# Day 28: Week 4 Review & System Design Mock

**Week 4 — System Design**

---

## 📋 Week 4 Cheat Sheet

| Concept | Definition / Use Case |
|---------|-----------------------|
| **Consistent Hashing** | Distributing data evenly across a cluster, minimizing re-hashing when nodes join/leave. |
| **CAP Theorem** | You can only have 2 of 3: Consistency, Availability, Partition Tolerance. |
| **Message Queues** | (Kafka/RabbitMQ) Decoupling heavy asynchronous tasks (like LLM Agent execution) from HTTP threads. |
| **Token Bucket** | Rate limiting algorithm allowing bursty traffic up to a max capacity, refilled at a constant rate. |
| **Idempotency** | Ensuring an API request can be safely retried multiple times without adverse side effects. |
| **TraceID / SpanID** | Distributed tracing identifiers to visualize bottlenecks across microservices. |
| **Canary Deployment** | Rolling out a new version to 1% of users to test for errors before a full release. |

## 🎯 Mock Interview Instructions

Today is your Week 4 execution test. A typical Google System Design interview lasts 45 minutes.
Below is a prompt. Do not read the solution until you have spent at least 35 minutes whiteboarding your architecture, defining your APIs, and listing your bottlenecks.

## 🕰️ Mock Prompt: Design an AI Code Review System

**The Problem**:
Design a system for GitHub/GitLab where every time a developer opens a Pull Request (PR), an AI Agent automatically reviews the code, leaves line-by-line comments for bugs or style issues, and assigns a "Pass/Fail" grade. 

**Scale**:
- 10 Million PRs opened per day globally.
- Average PR contains 5 files and 200 lines of code changed.
- The AI review should complete within 3 minutes of the PR being opened.

---

*Spend 35 minutes outlining your design before scrolling down!*

<br><br><br><br><br><br><br><br><br><br><br><br><br>

## 🛠️ Evaluation Rubric & Solution Architecture

Did you follow the 5-Step Framework?

### 1. Requirements Clarification
Did you ask about:
- **Security**: Can we send proprietary source code to a public LLM API, or must we host our own models?
- **Context**: Does the AI need the entire repository context, or just the diff of the PR?
- **Rate Limits**: Are we constrained by the LLM Provider's Tokens Per Minute (TPM)?

### 2. API Design
Instead of a standard REST API, this is an **Event-Driven** system. Did you identify that the trigger is a Webhook?
- `POST /webhooks/pr_opened`
  - Payload: `{pr_id, repo_id, diff_url, user_id}`

### 3. High-Level Architecture
Did you use a Message Queue? 
1. **Webhook Receiver**: A lightweight API Gateway receives the PR event and immediately pushes it to **Kafka** (to prevent timeout and handle spikes).
2. **Orchestrator Workers**: Consume the Kafka topics. They fetch the code Diff from the GitHub API.
3. **LLM Integration**: The worker chunks the code diff, constructs a prompt, and calls the LLM (e.g., Gemini).
4. **Feedback Loop**: Once the LLM returns its review JSON, the worker calls the GitHub API `POST /repos/{repo}/pulls/{pr}/comments` to post the results.

### 4. Deep Dive & Bottlenecks
Did you discuss these advanced topics?
- **Token Limits (Chunking)**: If a PR is 10,000 lines long, it might exceed the context window. Did you propose chunking the files, or using an LLM with a massive context window (like Gemini 1.5 Pro)?
- **Rate Limiting & Throttling**: 10 Million PRs/day = ~115 PRs/second. If every PR requires 5 LLM calls, that's 600 LLM calls/second. If your LLM provider enforces a strict quota, your workers will get HTTP 429s. Did you implement **Exponential Backoff** and a **Dead Letter Queue (DLQ)** for failed messages?
- **Caching**: If a developer updates a PR with a 1-line change to fix a typo, should we re-review the entire PR and waste LLM tokens? Did you suggest caching the hash of unchanged files so we only run the LLM on new diffs?

## 📝 Week 4 Self-Assessment

Rate yourself (1-5) on each area:

| Area | Rating | Notes |
|------|--------|-------|
| System Design Framework (The 5 Steps) | /5 | |
| Core Components (LB, API Gateway, CDN) | /5 | |
| Databases (SQL vs NoSQL, Sharding) | /5 | |
| Event-Driven Architectures (Queues/Kafka)| /5 | |
| API Design & Rate Limiting | /5 | |
| Observability (Metrics, Tracing, SLOs) | /5 | |
| Deployment (K8s, CI/CD, Canary) | /5 | |

### Ready for Week 5? ✅
Week 4 shifted our mindset from writing algorithms to architecting systems. 
Next week (`week-05-agent-development`), we dive into the exact domain you are interviewing for. We will write actual code using LangChain/Agno, build tools, manage context windows, and implement RAG (Retrieval-Augmented Generation) systems!